Create New Train Env

In [ ]:
%pip install "gymnasium[box2d]" stable-baselines3 matplotlib pandas

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gymnasium as gym

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.utils import get_linear_fn

Create Project Folders

In [ ]:
BASE_DIR = "bipedalwalker_project"
LOG_DIR = os.path.join(BASE_DIR, "logs")
MODEL_DIR = os.path.join(BASE_DIR, "models")
BEST_MODEL_DIR = os.path.join(BASE_DIR, "best_model")
VIDEO_DIR = os.path.join(BASE_DIR, "videos")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(VIDEO_DIR, exist_ok=True)

print("Folders created successfully.")

Create New Eval Evn

In [ ]:
ENV_ID = "BipedalWalker-v3"
N_ENVS = 8

train_env = make_vec_env(
    ENV_ID,
    n_envs=N_ENVS,
    monitor_dir=LOG_DIR
)

eval_env = Monitor(gym.make(ENV_ID))

print("Training and evaluation environments created.")

Wrap/ Log Env

Def PPO Hyperparameters 

In [ ]:
model = PPO(
    policy="MlpPolicy",
    env=train_env,
    verbose=1,
    learning_rate=get_linear_fn(
        3e-4,
        1e-5,
        1.0
    ),
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.001,
    policy_kwargs=dict(
        net_arch=[dict(
            pi=[256, 256],
            vf=[256, 256]
        )]
    ),
    tensorboard_log=os.path.join(BASE_DIR, "tensorboard")
)

print("PPO model created.")

Create Callbacks and checkpoints 

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq=50_000 // N_ENVS,
    save_path=MODEL_DIR,
    name_prefix="ppo_bipedalwalker_checkpoint"
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=BEST_MODEL_DIR,
    log_path=BEST_MODEL_DIR,
    eval_freq=25_000 // N_ENVS,
    deterministic=True,
    render=False
)

callback = CallbackList([checkpoint_callback, eval_callback])

print("Callbacks ready.")

TRAIN

In [ ]:
TOTAL_TIMESTEPS = 5_000_000

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=callback,
    progress_bar=True
)

FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "ppo_bipedalwalker_final")
model.save(FINAL_MODEL_PATH)

print(f"Training complete. Final model saved to: {FINAL_MODEL_PATH}")

Save CheckPoints 

Eval Periodically

In [ ]:
mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"Final model mean reward: {mean_reward:.2f}")
print(f"Final model standard deviation: {std_reward:.2f}")

In [ ]:
best_model_path = os.path.join(BEST_MODEL_DIR, "best_model.zip")

best_model = PPO.load(best_model_path)

best_mean_reward, best_std_reward = evaluate_policy(
    best_model,
    eval_env,
    n_eval_episodes=10,
    deterministic=True
)

print(f"Best model mean reward: {best_mean_reward:.2f}")
print(f"Best model standard deviation: {best_std_reward:.2f}")

Plot results

In [ ]:
def load_monitor_files(log_dir):
    csv_files = glob.glob(os.path.join(log_dir, "*.monitor.csv"))
    dataframes = []

    for file in csv_files:
        df = pd.read_csv(file, skiprows=1)
        dataframes.append(df)

    if not dataframes:
        raise FileNotFoundError("No monitor files found in the log directory.")

    combined_df = pd.concat(dataframes, ignore_index=True)
    return combined_df

monitor_df = load_monitor_files(LOG_DIR)

print("Monitor logs loaded successfully.")
print(monitor_df.head())

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monitor_df["r"].values)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("BipedalWalker Training Reward per Episode")
plt.grid(True)
plt.show()

In [ ]:
window = 50

monitor_df["reward_smooth"] = monitor_df["r"].rolling(window=window).mean()

print(monitor_df[["r", "reward_smooth"]].head(10))

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monitor_df["reward_smooth"].values)
plt.xlabel("Episode")
plt.ylabel(f"Reward (Rolling Mean, window={window})")
plt.title("Smoothed BipedalWalker Training Reward")
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(monitor_df["r"].values, alpha=0.3, label="Raw Reward")

plt.plot(monitor_df["reward_smooth"].values, linewidth=2, label="Smoothed Reward")

plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("BipedalWalker PPO Learning Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
monitor_df["cumulative_timesteps"] = monitor_df["l"].cumsum()

plt.figure(figsize=(12, 6))
plt.plot(monitor_df["cumulative_timesteps"], monitor_df["r"], alpha=0.3, label="Raw Reward")

plt.plot(monitor_df["cumulative_timesteps"], monitor_df["reward_smooth"], linewidth=2, label="Smoothed Reward")

plt.xlabel("Timesteps")
plt.ylabel("Reward")
plt.title("BipedalWalker Reward vs Timesteps")
plt.legend()
plt.grid(True)
plt.show()

Record Final Run

In [ ]:
watch_env = gym.make(ENV_ID, render_mode="human")
obs, info = watch_env.reset()

for step in range(2000):
    action, _states = model.predict(obs, deterministic=True)

    obs, reward, terminated, truncated, info = watch_env.step(action)

    if terminated or truncated:
        obs, info = watch_env.reset()

watch_env.close()